<a href="https://colab.research.google.com/github/imtisalrangrez/DeepLearning_Lab/blob/main/DL_week7(180).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Q18:LE-NET 5
Adequate but Not Perfect: LeNet-5 was invented in the 1990s specifically to read simple handwritten digits (like postal codes). Fashion MNIST has much more complex textures and edges (e.g., distinguishing a pullover from a long-sleeve shirt). The model will achieve roughly 88% to 89% accuracy, which is good, but modern architectures easily break 92%+.

tanh vs. relu: This original architecture uses tanh activations. You will notice that training starts off a bit slower than it would if you used the modern relu activation function, because tanh gradients can vanish on the edges.

Overfitting Risk: By epoch 15-20, you will likely notice the loss continuing to drop while the val_loss (validation loss) levels out or slightly increases. Because classic LeNet-5 does not include Dropout layers, it tends to memorize the training images slightly toward the end of the run.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Load CIFAR-10
# ---------------------------------------------------------
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

# ---------------------------------------------------------
# 2. Visualize Training Data
# ---------------------------------------------------------
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])          # No cmap — images are RGB color
    plt.title(class_names[y_train[i][0]])
    plt.axis('off')
plt.suptitle("CIFAR-10 Sample Images", fontsize=14)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 3. Data Preprocessing
# ---------------------------------------------------------
# CIFAR-10 is already (batch, 32, 32, 3) — no reshape needed
x_train = x_train / 255.0
x_test  = x_test  / 255.0

# ---------------------------------------------------------
# 4. Build LeNet-5 Model (adapted for CIFAR-10)
# ---------------------------------------------------------
LeNet5_Model = tf.keras.Sequential([
    tf.keras.Input(shape=(32, 32, 3)),

    # Block 1 — padding='same' preserves spatial size before pooling
    tf.keras.layers.Conv2D(6, kernel_size=(5, 5), activation='relu', padding='same'),
    tf.keras.layers.AveragePooling2D((2, 2)),           # → 16x16

    # Block 2
    tf.keras.layers.Conv2D(16, kernel_size=(5, 5), activation='relu', padding='same'),
    tf.keras.layers.AveragePooling2D((2, 2)),           # → 8x8

    # Classification Head — larger Dense layers for CIFAR-10's complexity
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    # tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(128, activation='relu'),
    # tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10, activation='softmax')    # 10 CIFAR-10 classes
])

# ---------------------------------------------------------
# 5. Compile The Model
# ---------------------------------------------------------
LeNet5_Model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

LeNet5_Model.summary()

# ---------------------------------------------------------
# 6. Train The Model
# ---------------------------------------------------------
# validation_split=0.1666 → ~8,333 samples used for validation internally
print("\n--- Starting Training ---")
history = LeNet5_Model.fit(
    x_train, y_train,
    epochs=20,
    batch_size=128,
    validation_split=0.1666
)

# ---------------------------------------------------------
# 7. Evaluate On Test Dataset
# ---------------------------------------------------------
print("\n--- Evaluating Model ---")
test_loss, test_acc = LeNet5_Model.evaluate(x_test, y_test)
print(f'Test Accuracy : {test_acc:.4f}')
print(f'Test Loss     : {test_loss:.4f}')

# Expected results (20 epochs, no dropout):  ~65-68% accuracy
# Expected results (20 epochs, with dropout): ~62-65% accuracy
# Note: LeNet-5 is a shallow architecture — CIFAR-10 genuinely needs deeper
# nets (VGG, ResNet) to push past ~75%. This is a great baseline to compare against.

#Alex-NET

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential, layers
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Load and Preprocess CIFAR-10
# ---------------------------------------------------------
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['Airplane','Automobile','Bird','Cat','Deer',
               'Dog','Frog','Horse','Ship','Truck']

# Normalize — already (batch, 32, 32, 3), no reshape needed
x_train = x_train / 255.0
x_test  = x_test  / 255.0

# ---------------------------------------------------------
# 2. Build Adapted AlexNet for CIFAR-10
# ---------------------------------------------------------
model = Sequential([
    tf.keras.Input(shape=(32, 32, 3)),

    # Layer 1: Slightly larger kernel to capture richer color features
    layers.Conv2D(96, (5, 5), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),                        # → 16x16

    # Layer 2
    layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),                        # → 8x8

    # Layers 3, 4, 5: Stacked convolutions — no pooling here to preserve 8x8 spatial info
    layers.Conv2D(384, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(384, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),                        # → 4x4

    # Flatten + Fully Connected (scaled down from original 4096)
    layers.Flatten(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.5),

    # Output — 10 CIFAR-10 classes
    layers.Dense(10, activation='softmax')
])

# ---------------------------------------------------------
# 3. Compile
# ---------------------------------------------------------
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ---------------------------------------------------------
# 4. Train
# ---------------------------------------------------------
print("\n--- Starting Training ---")
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1
)

# ---------------------------------------------------------
# 5. Evaluate
# ---------------------------------------------------------
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f'\nTest Accuracy : {test_acc:.4f}')
print(f'Test Loss     : {test_loss:.4f}')

# Expected: ~78-82% accuracy (10 epochs)
# LeNet-5 on CIFAR-10 baseline was ~65-68% — AlexNet's depth gives a clear boost

# ---------------------------------------------------------
# 6. Plot Accuracy and Loss
# ---------------------------------------------------------
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title('Accuracy over Epochs — CIFAR-10 AlexNet')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Loss over Epochs — CIFAR-10 AlexNet')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.tight_layout()
plt.show()

#VGG NET

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential, layers
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Load and Preprocess CIFAR-10
# ---------------------------------------------------------
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['Airplane','Automobile','Bird','Cat','Deer',
               'Dog','Frog','Horse','Ship','Truck']

# Normalize — already (batch, 32, 32, 3), no reshape needed
x_train = x_train / 255.0
x_test  = x_test  / 255.0

# ---------------------------------------------------------
# 2. Build Adapted VGG for CIFAR-10
# ---------------------------------------------------------
model = Sequential([
    tf.keras.Input(shape=(32, 32, 3)),

    # VGG Block 1: 64 filters — more capacity for RGB feature extraction
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),                        # → 16x16

    # VGG Block 2: Filters double to 128
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),                        # → 8x8

    # VGG Block 3: Three convs at 256 — deeper block for complex spatial features
    layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),                        # → 4x4

    # Classification Head — larger than Fashion MNIST version
    layers.Flatten(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')              # 10 CIFAR-10 classes
])

# ---------------------------------------------------------
# 3. Compile
# ---------------------------------------------------------
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ---------------------------------------------------------
# 4. Train with Early Stopping
# ---------------------------------------------------------
# patience=4 gives CIFAR-10 more room to improve before stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

print("\n--- Starting VGG Training on CIFAR-10 ---")
history = model.fit(
    x_train, y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop]
)

# ---------------------------------------------------------
# 5. Evaluate
# ---------------------------------------------------------
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f'\nFinal Test Accuracy : {test_acc:.4f}')
print(f'Final Test Loss     : {test_loss:.4f}')

# Expected: ~83-87% accuracy
# Comparison so far on CIFAR-10:
#   LeNet-5  → ~65-68%
#   AlexNet  → ~78-82%
#   VGG      → ~83-87%   ← deeper blocks + dropout = clear improvement

# ---------------------------------------------------------
# 6. Plot Learning Curves
# ---------------------------------------------------------
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.legend()
plt.title('VGG Accuracy — CIFAR-10')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('VGG Loss — CIFAR-10')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.tight_layout()
plt.show()